In [372]:
import qiskit
from qiskit_algorithms import QAOA
from qiskit_optimization import QuadraticProgram

## Test examples :

In [373]:
W = [
    [0, 0, 0, 0],
    [0, 0, 50, 5],
    [0, 50, 0, 50],
    [0, 5, 50, 0]
]

# W = [
#     [0, 10, 1],
#     [10, 0, 3],
#     [1, 3, 0],
# ]


# W = [
#     [0, 2],
#     [2, 0]
# ]

# W = [
#     [  0.00,  26.57,  12.07,  47.19, 126.81,  35.69,  77.42,  76.09, 125.47,  78.93],
#     [ 40.15,   0.00,  41.21,  78.54, 118.78,  41.20,  91.83,  85.67, 134.36, 140.95],
#     [ 12.07,  70.52,   0.00,  39.15,  64.20,  92.26, 115.77, 117.75, 141.45,  39.78],
#     [ 62.83,  40.30,  39.15,   0.00,  48.47,  15.35,  96.51,  49.51,  90.94,   0.77],
#     [172.35, 134.38,  64.20,  27.94,   0.00,  48.81,  26.91,  23.45,  31.52,  28.75],
#     [ 90.54,  41.20,  42.21,   3.04,  49.72,   0.00,  68.40,  48.17, 117.34,   8.62],
#     [ 76.64,  89.36,  87.14,  52.94,  26.91,  94.27,   0.00,  12.55,   8.96,  67.32],
#     [ 77.27,  85.52,  87.38,  52.82,  23.46,  48.17,   6.24,   0.00,   9.16,  67.09],
#     [130.99, 133.05, 110.34,  73.83,  31.52,  83.61,  11.19,   8.96,   0.00,  58.48],
#     [113.36,  74.74,  40.17,   0.77,  38.26,   8.56, 134.50,  65.68,  58.58,   0.00],
# ]

# W = W[:4][:4]

V = [0, 20, 30, 20]

L = 4


n = len(W)

print(f"Number of cities : {n}")
print(f"Number of variables : {n**2}")
print(f"Size of hamiltonian : {2**(n**2)}")



Number of cities : 4
Number of variables : 16
Size of hamiltonian : 65536


## Old Formulation :

In [374]:
# qp = QuadraticProgram(name="TSP-City-Selection")

# # --- Variables: only x[i][p] ---
# for i in range(n):
#     for p in range(n):
#         qp.binary_var(name=f"x_{i}_{p}")

# def x(i, p): return f"x_{i}_{p}"

# # --- Penalty scalar ---
# P = max(V)/L
# print(P)
# # "each unit of fuel is worth max_value/L in penalty"
# # routes exceeding L become naturally unattractive

# # --- Objective: maximize value - penalty for excess fuel ---
# # maximize sum_i V[i] * sum_p x[i][p]
# # minus P * max(0, fuel - L)
# # but max(0, ...) is non-linear, so instead penalize all fuel directly:
# # maximize sum V[i]*x[i][p] - P * sum W[i][j]*x[i][p]*x[j][p+1]
# # which naturally discourages high fuel routes

# linear_terms = {}
# for i in range(n):
#     for p in range(n):
#         linear_terms[x(i, p)] = V[i]

# quadratic_terms = {}
# for i in range(n):
#     for j in range(n):
#         if i != j:
#             for p in range(n - 1):
#                 key = (x(i, p), x(j, p + 1))
#                 quadratic_terms[key] = - P * W[i][j] # quadratic_terms.get(key, 0)

# qp.maximize(linear=linear_terms, quadratic=quadratic_terms)

# # --- Constraint 1: each city visited AT MOST once ---
# for i in range(n):
#     qp.linear_constraint(
#         linear={x(i, p): 1 for p in range(n)},
#         sense="<=",
#         rhs=1,
#         name=f"city_{i}_at_most_once"
#     )

# # --- Constraint 2: each position has AT MOST one city ---
# for p in range(n):
#     qp.linear_constraint(
#         linear={x(i, p): 1 for i in range(n)},
#         sense="<=",
#         rhs=1,
#         name=f"position_{p}_at_most_one"
#     )

# # --- Constraint 3: positions must be contiguous ---
# for p in range(1, n):
#     qp.linear_constraint(
#         linear={**{x(i, p): 1 for i in range(n)},
#                 **{x(i, p-1): -1 for i in range(n)}},
#         sense="<=",
#         rhs=0,
#         name=f"contiguous_positions_{p}"
#     )

# print(qp.export_as_lp_string())

## New formulation :

In [375]:
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

# W_max = max(W[i][j] for i in range(n) for j in range(n) if i!=j)
# V_max = max(V)

# W_norm = [[W[i][j] / W_max for j in range(n)] for i in range(n)]
# V_norm = [v / V_max for v in V]
# L_norm = L / W_max

# W = W_norm
# V = V_norm
# L = L_norm

qp = QuadraticProgram(name="TSP-City-Selection")

def x(i,j): return f"x_{i}_{j}"

# --- Variables: edge-based ---
for i in range(n):
    for j in range(n):
        if i != j:
            qp.binary_var(name=x(i,j))
                        
P = (max(V) / max(W[i][j] for i in range(n) for j in range(n) if i != j))*2
print(P)

linear_terms = {}

# Value reward via in-degree
for i in range(1, n):
    for j in range(n):
        if i != j:
            linear_terms[x(j,i)] = linear_terms.get(x(j,i), 0) + V[i]

# Linear fuel penalty
for i in range(n):
    for j in range(n):
        if i != j:
            linear_terms[x(i,j)] = linear_terms.get(x(i,j), 0) - P * W[i][j]

qp.maximize(linear=linear_terms)



# --- C0: out-degree of 0 = 1 ---

qp.linear_constraint(
    linear={x(0,j): 1 for j in range(n) if j!=0},
    sense="==", rhs=1,
    name=f"startout_{0}"
)
qp.linear_constraint(
    linear={x(j,0): 1 for j in range(n) if j!=0},
    sense="==", rhs=1,
    name=f"startin_{0}"
)

# --- C1: out-degree <= 1 ---
for i in range(1, n):
    qp.linear_constraint(
        linear={x(i,j): 1 for j in range(n) if j!=i},
        sense="<=", rhs=1,
        name=f"out_{i}"
    )

# --- C2: in-degree <= 1 ---
for i in range(1, n):
    qp.linear_constraint(
        linear={x(j,i): 1 for j in range(n) if j!=i},
        sense="<=", rhs=1,
        name=f"in_{i}"
    )

# --- C3: flow conservation ---
for i in range(n):
    qp.linear_constraint(
        linear={**{x(i,j):  1 for j in range(n) if j!=i},
                **{x(j,i): -1 for j in range(n) if j!=i}},
        sense="==", rhs=0,
        name=f"flow_{i}"
    )

from itertools import combinations

# --- C4: no subtour on subsets not containing vertex 0 ---
# For every subset S of {1,...,n-2} with |S| >= 2,
# the number of directed edges within S must be <= |S| - 1
for size in range(2, n - 1):                          # subsets of size 2 to n-1
    for S in combinations(range(1, n), size):     # subsets of {1,...,n-1}
        qp.linear_constraint(
            linear={x(i,j): 1 for i in S for j in S if i != j},
            sense="<=",
            rhs=len(S) - 1,
            name=f"no_subtour_{'_'.join(map(str, S))}"
        )

print(qp.export_as_lp_string())

# --- Convert to QUBO and check qubit count ---
converter = QuadraticProgramToQubo()
qubo = converter.convert(qp)

print("\n--- Qubit count ---")
print(f"Problem variables:  {qp.get_num_vars()}")
print(f"QUBO variables:     {qubo.get_num_vars()}")
print(f"Slack qubits added: {qubo.get_num_vars() - qp.get_num_vars()}")
print(f"\nQUBO variable names: {[v.name for v in qubo.variables]}")

1.2
\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: TSP-City-Selection

Maximize
 obj: 20 x_0_1 + 30 x_0_2 + 20 x_0_3 - 30 x_1_2 + 14 x_1_3 - 40 x_2_1 - 40 x_2_3
      + 14 x_3_1 - 30 x_3_2
Subject To
 startout_0: x_0_1 + x_0_2 + x_0_3 = 1
 startin_0: x_1_0 + x_2_0 + x_3_0 = 1
 out_1: x_1_0 + x_1_2 + x_1_3 <= 1
 out_2: x_2_0 + x_2_1 + x_2_3 <= 1
 out_3: x_3_0 + x_3_1 + x_3_2 <= 1
 in_1: x_0_1 + x_2_1 + x_3_1 <= 1
 in_2: x_0_2 + x_1_2 + x_3_2 <= 1
 in_3: x_0_3 + x_1_3 + x_2_3 <= 1
 flow_0: x_0_1 + x_0_2 + x_0_3 - x_1_0 - x_2_0 - x_3_0 = 0
 flow_1: - x_0_1 + x_1_0 + x_1_2 + x_1_3 - x_2_1 - x_3_1 = 0
 flow_2: - x_0_2 - x_1_2 + x_2_0 + x_2_1 + x_2_3 - x_3_2 = 0
 flow_3: - x_0_3 - x_1_3 - x_2_3 + x_3_0 + x_3_1 + x_3_2 = 0
 no_subtour_1_2: x_1_2 + x_2_1 <= 1
 no_subtour_1_3: x_1_3 + x_3_1 <= 1
 no_subtour_2_3: x_2_3 + x_3_2 <= 1

Bounds
 0 <= x_0_1 <= 1
 0 <= x_0_2 <= 1
 0 <= x_0_3 <= 1
 0 <= x_1_0 <= 1
 0 <= x_1_2 <= 1
 0 <= x_1_3 <= 1
 0 <= x_2_0 <= 1
 0 <= x

## Convert to QUBO

In [376]:
from qiskit_optimization.converters import (
    QuadraticProgramToQubo,   # folds constraints into objective via penalties
    LinearEqualityToPenalty,  # sub-step: converts == constraints to penalty terms
    MinimizeToMaximize,       # flips sign if needed
    IntegerToBinary,          # converts integer vars to binary (not needed here)
)
from qiskit_optimization.translators import to_ising
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
import numpy as np

max_reward = sum(V) * n  # worst case all cities visited
penalty = max_reward * 10  # safely above any possible reward

# Step 1: QP → QUBO  
qubo_converter = QuadraticProgramToQubo()
qubo = qubo_converter.convert(qp)
print(qubo)

# Step 2: QUBO → Ising Hamiltonian 
ising_operator, offset = to_ising(qubo)


minimize 15774*x_0_1^2 + 21032*x_0_1*x_0_2 + 21032*x_0_1*x_0_3 - 21032*x_0_1*x_1_0 - 10516*x_0_1*x_1_2 - 10516*x_0_1*x_1_3 - 10516*x_0_1*x_2_0 + 10755*x_0_1*x_2_1 - 10516*x_0_1*x_3_0 + 10755*x_0_1*x_3_1 + 15774*x_0_2^2 + 21032*x_0_2*x_0_3 - 10516*x_0_2*x_1_0 + 10755*x_0_2*x_1_2 - 21032*x_0_2*x_2_0 - 10516*x_0_2*x_2_1 - 10516*x_0_2*x_2_3 - 10516*x_0_2*x_3_0 + 10755*x_0_2*x_3_2 + 15774*x_0_3^2 - 10516*x_0_3*x_1_0 + 10755*x_0_3*x_1_3 - 10516*x_0_3*x_2_0 + 10755*x_0_3*x_2_3 - 21032*x_0_3*x_3_0 - 10516*x_0_3*x_3_1 - 10516*x_0_3*x_3_2 + 15774*x_1_0^2 + 10755*x_1_0*x_1_2 + 10755*x_1_0*x_1_3 + 21032*x_1_0*x_2_0 - 10516*x_1_0*x_2_1 + 21032*x_1_0*x_3_0 - 10516*x_1_0*x_3_1 + 10516*x_1_2^2 + 10755*x_1_2*x_1_3 - 10516*x_1_2*x_2_0 - 20793*x_1_2*x_2_1 - 10516*x_1_2*x_2_3 - 10516*x_1_2*x_3_1 + 10755*x_1_2*x_3_2 + 10516*x_1_3^2 - 10516*x_1_3*x_2_1 + 10755*x_1_3*x_2_3 - 10516*x_1_3*x_3_0 - 20793*x_1_3*x_3_1 - 10516*x_1_3*x_3_2 + 15774*x_2_0^2 + 10755*x_2_0*x_2_1 + 10755*x_2_0*x_2_3 + 21032*x_2_0*x_3_0 -

In [377]:
# Find the smallest absolute coefficient in the QUBO

linear_dict    = qubo.objective.linear.to_dict(use_name=True)
quadratic_dict = qubo.objective.quadratic.to_dict(use_name=True)

all_coeffs = list(linear_dict.values()) + list(quadratic_dict.values())
min_coeff  = min(abs(c) for c in all_coeffs)
scale      = min_coeff

print(scale)

# Step 3: rebuild scaled QUBO
qubo_scaled = QuadraticProgram(name=f"{qp.name}-scaled")
for v in qubo.variables:
    qubo_scaled.binary_var(name=v.name)

qubo_scaled.minimize(
    linear    = {k: v / scale for k, v in linear_dict.items()},
    quadratic = {k: v / scale for k, v in quadratic_dict.items()}
)

print(f"Max coefficient after scaling: {min_coeff / scale}")
print(qubo_scaled)

14.0
Max coefficient after scaling: 1.0
minimize 1126.7142857142858*x_0_1^2 + 1502.2857142857142*x_0_1*x_0_2 + 1502.2857142857142*x_0_1*x_0_3 - 1502.2857142857142*x_0_1*x_1_0 - 751.1428571428571*x_0_1*x_1_2 - 751.1428571428571*x_0_1*x_1_3 - 751.1428571428571*x_0_1*x_2_0 + 768.2142857142857*x_0_1*x_2_1 - 751.1428571428571*x_0_1*x_3_0 + 768.2142857142857*x_0_1*x_3_1 + 1126.7142857142858*x_0_2^2 + 1502.2857142857142*x_0_2*x_0_3 - 751.1428571428571*x_0_2*x_1_0 + 768.2142857142857*x_0_2*x_1_2 - 1502.2857142857142*x_0_2*x_2_0 - 751.1428571428571*x_0_2*x_2_1 - 751.1428571428571*x_0_2*x_2_3 - 751.1428571428571*x_0_2*x_3_0 + 768.2142857142857*x_0_2*x_3_2 + 1126.7142857142858*x_0_3^2 - 751.1428571428571*x_0_3*x_1_0 + 768.2142857142857*x_0_3*x_1_3 - 751.1428571428571*x_0_3*x_2_0 + 768.2142857142857*x_0_3*x_2_3 - 1502.2857142857142*x_0_3*x_3_0 - 751.1428571428571*x_0_3*x_3_1 - 751.1428571428571*x_0_3*x_3_2 + 1126.7142857142858*x_1_0^2 + 768.2142857142857*x_1_0*x_1_2 + 768.2142857142857*x_1_0*x_1_3

## Instanciate AERSampler

In [378]:
from qiskit_aer.primitives import Sampler as AerSampler

sampler = AerSampler(
    backend_options={
        "method": "statevector", #matrix_product_state
        "max_parallel_threads": 0,      # 0 = all cores
        "max_parallel_experiments": 0,
        "max_parallel_shots": 0
    },
    run_options={
        "shots": 1024
    }
)

optimizer = COBYLA()

## Solve QAOA

In [379]:

# Step 3: feed to QAOA
qaoa = QAOA(
    sampler=sampler,
    optimizer=optimizer,
    reps=5,               # p layers — more layers = better approximation, more circuit depth
    initial_point=None,   # optional initial [β, γ] parameters, length = 2p
    mixer=None,           # default X mixer; can provide custom circuit for constrained problems
)
result = qaoa.compute_minimum_eigenvalue(ising_operator)
print(result)


# Step 4: interpret result back in terms of original variables
bitstring = result.best_measurement["bitstring"]           # e.g. "1000010000100001"
x_array   = np.array([int(b) for b in bitstring])         # → [1,0,0,0,0,1,0,0,...]

decoded = qubo_converter.interpret(x_array)
print(decoded)

{   'aux_operators_evaluated': None,
    'best_measurement': {   'bitstring': '001000100001',
                            'probability': 0.001953125,
                            'state': 545,
                            'value': np.complex128(-43373.75+0j)},
    'cost_function_evals': np.int64(79),
    'eigenstate': {2372: 0.0009765625, 618: 0.0009765625, 3704: 0.0009765625, 3346: 0.0009765625, 3648: 0.001953125, 2110: 0.0009765625, 3681: 0.0009765625, 3360: 0.0009765625, 2149: 0.0009765625, 1259: 0.0009765625, 3873: 0.0009765625, 1708: 0.0009765625, 862: 0.0009765625, 2145: 0.0009765625, 2178: 0.0009765625, 229: 0.0009765625, 1292: 0.0009765625, 3531: 0.0009765625, 3920: 0.0009765625, 3537: 0.0009765625, 3937: 0.0009765625, 1622: 0.0009765625, 170: 0.0009765625, 2713: 0.0009765625, 3434: 0.0009765625, 2904: 0.0009765625, 2051: 0.0009765625, 1560: 0.0009765625, 1417: 0.0009765625, 1537: 0.0009765625, 2992: 0.0009765625, 3343: 0.0009765625, 1543: 0.0009765625, 1857: 0.0009765625, 264: 0

## Convert bitstring to path

In [380]:
def edges_to_path(x_array, n):
    """
    x_array: flat binary array from QAOA, ordered as x_0_1, x_0_2, ..., x_i_j (skipping i==j)
    """
    # Convert flat binary array to list of (i,j) edge tuples
    edges_used = []
    idx = 0
    for i in range(n):
        for j in range(n):
            if i != j:
                if x_array[idx] == 1:
                    edges_used.append((i, j))
                idx += 1
    print(edges_used)

    if len(edges_used) == 0:
        print("No edges used")
        return []

    # Build adjacency from used edges
    next_city  = {i: j for i, j in edges_used}
    in_cities  = {j for i, j in edges_used}
    out_cities = {i for i, j in edges_used}

    # Source = city with outgoing edge but no incoming edge
    sources = out_cities - in_cities

    if not sources:
        print("No valid path found (possibly a cycle)")
        return []

    # Follow the path from source
    path = []
    current = next(iter(sources))
    while current in next_city:
        path.append(current)
        current = next_city[current]
    path.append(current)  # append sink

    print(" -> ".join(str(c) for c in path))
    return path

# Usage
edges_to_path(decoded[::-1], n)

[(0, 1), (1, 3), (3, 0)]
No valid path found (possibly a cycle)


[]